# Master — MOZAK (nnU-Net)
Priprema podataka (sve lezije) i prebacivanje u format koji nnU-Net zahteva, pa pokretanje nnU-Net planiranje i pretprocesiranje.


## Ćelija 0 — PRIPREMA

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q nnunetv2 nibabel

import os, shutil, json, random
import numpy as np
import nibabel as nib

STARO = '/content/drive/MyDrive/ms projekat'
MASTER = '/content/drive/MyDrive/Master/mozak'

os.environ['nnUNet_raw']          = f'{MASTER}/podaci_nnunet/nnUNet_raw'
os.environ['nnUNet_preprocessed'] = '/content/nnUNet_preprocessed'
os.environ['nnUNet_results']      = f'{MASTER}/modeli/nnUNet_results'

for p in [os.environ['nnUNet_raw'], os.environ['nnUNet_preprocessed'], os.environ['nnUNet_results']]:
    os.makedirs(p, exist_ok=True)

import torch
print('GPU dostupan:', torch.cuda.is_available(), '(za pripremu nije potreban)')
print('nnUNet_raw:', os.environ['nnUNet_raw'])
print('Spremno.')


Mounted at /content/drive
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 291.1/291.1 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.8/106.8 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.6/53.6 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.4/28.4 MB 74.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 95.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 122.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 10.4 MB/s eta 0:00:00
GPU dostupan: True (za pripremu nije potreban)
nn

## Ćelija 1 — provera MS3SEG na Drive-u


In [ ]:
s_dir = f'{STARO}/ms3seg/snimci'
m_dir = f'{STARO}/ms3seg/maske'

ms3 = []
for pid in sorted(os.listdir(s_dir)):
    s = f'{s_dir}/{pid}/{pid}_FLAIR.nii.gz'
    m = f'{m_dir}/{pid}/{pid}_abWMH_Mask.nii.gz'
    if os.path.exists(s) and os.path.exists(m):
        ms3.append({'id': f'ms3_{pid}', 'img': s, 'lab': m})

print('MS3SEG pacijenata nađeno:', len(ms3))
assert len(ms3) > 0, 'MS3SEG nije nađen! Proveri da je ms projekat/ms3seg/snimci na Drive-u.'

pids = sorted([d['id'] for d in ms3])
random.seed(42); random.shuffle(pids)
test_ids = set(pids[85:])     # 15 test
ms3_tr = [d for d in ms3 if d['id'] not in test_ids]
ms3_ts = [d for d in ms3 if d['id'] in test_ids]
print('MS3SEG trening:', len(ms3_tr), '| MS3SEG interni test:', len(ms3_ts))


MS3SEG pacijenata nađeno: 100
MS3SEG trening: 85 | MS3SEG interni test: 15


## Ćelija 2 — preuzimanje

In [ ]:
import requests, zipfile

if not os.path.exists('/content/mslesseg/MSLesSeg Dataset'):
    url = 'https://ndownloader.figshare.com/files/52771814'
    print('Skidam MSLesSeg (1.47 GB, potraje)...')
    r = requests.get(url)
    with open('/content/mslesseg.zip', 'wb') as f:
        f.write(r.content)
    print('Skinuto, raspakivam...')
    with zipfile.ZipFile('/content/mslesseg.zip', 'r') as z:
        z.extractall('/content/mslesseg')
    print('Gotovo.')
else:
    print('MSLesSeg već raspakovan, preskačem.')

base_mls = '/content/mslesseg/MSLesSeg Dataset'

mls_tr = []
for pid in sorted(os.listdir(f'{base_mls}/train')):
    pdir = f'{base_mls}/train/{pid}'
    if not os.path.isdir(pdir): continue
    for t in sorted(os.listdir(pdir)):
        tdir = f'{pdir}/{t}'
        if not os.path.isdir(tdir): continue
        f_ = f'{tdir}/{pid}_{t}_FLAIR.nii.gz'
        m_ = f'{tdir}/{pid}_{t}_MASK.nii.gz'
        if os.path.exists(f_) and os.path.exists(m_):
            mls_tr.append({'id': f'mls_{pid}_{t}', 'img': f_, 'lab': m_})

mls_ts = []
tdir = f'{base_mls}/test'
if os.path.isdir(tdir):
    for pac in sorted(os.listdir(tdir)):
        f_ = f'{tdir}/{pac}/{pac}_FLAIR.nii.gz'
        m_ = f'{tdir}/{pac}/{pac}_MASK.nii.gz'
        if os.path.exists(f_) and os.path.exists(m_):
            mls_ts.append({'id': f'mls_{pac}', 'img': f_, 'lab': m_})

print('MSLesSeg trening slučajeva:', len(mls_tr), '| MSLesSeg test:', len(mls_ts))


Skidam MSLesSeg (1.47 GB, potraje)...
Skinuto, raspakivam...
Gotovo.
MSLesSeg trening slučajeva: 93 | MSLesSeg test: 22


## Ćelija 3 — pravljenje nnU-Net raw formata (Dataset001_MsBrain)
Spajanje MS3SEG (trening) + MSLesSeg (trening) u nnU-Net trening skup. MS3SEG interni test i MSLesSeg test idu u `imagesTs` (za evaluaciju).

In [ ]:
DS = os.environ['nnUNet_raw'] + '/Dataset001_MsBrain'
for sub in ['imagesTr','labelsTr','imagesTs','labelsTs']:
    os.makedirs(f'{DS}/{sub}', exist_ok=True)

def upisi_sliku(src, dst):
    shutil.copy(src, dst)

def upisi_masku(src, dst):
    img = nib.load(src)
    d = (np.asarray(img.dataobj) > 0).astype(np.uint8)
    nib.save(nib.Nifti1Image(d, img.affine, img.header), dst)

def napravi(skup, img_dir, lab_dir):
    n = 0
    for d in skup:
        cid = d['id']
        try:
            upisi_sliku(d['img'], f'{DS}/{img_dir}/{cid}_0000.nii.gz')
            upisi_masku(d['lab'], f'{DS}/{lab_dir}/{cid}.nii.gz')
            n += 1
        except Exception as e:
            print('  GREŠKA na', cid, '->', e)
    return n

print('Upisujem TRENING (MS3SEG + MSLesSeg)...')
n_tr  = napravi(ms3_tr + mls_tr, 'imagesTr', 'labelsTr')
print('  upisano trening slučajeva:', n_tr)

print('Upisujem TEST (MS3SEG interni + MSLesSeg)...')
n_ts  = napravi(ms3_ts + mls_ts, 'imagesTs', 'labelsTs')
print('  upisano test slučajeva:', n_ts)

dataset = {
    "channel_names": {"0": "FLAIR"},
    "labels": {"background": 0, "lesion": 1},
    "numTraining": n_tr,
    "file_ending": ".nii.gz",
    "overwrite_image_reader_writer": "SimpleITKIO"
}
with open(f'{DS}/dataset.json','w') as f:
    json.dump(dataset, f, indent=2)
print('dataset.json upisan. numTraining =', n_tr)


Upisujem TRENING (MS3SEG + MSLesSeg)...
  upisano trening slučajeva: 178
Upisujem TEST (MS3SEG interni + MSLesSeg)...
  upisano test slučajeva: 37
dataset.json upisan. numTraining = 178


## Ćelija 4

In [ ]:
imgs = sorted(os.listdir(f'{DS}/imagesTr'))
labs = sorted(os.listdir(f'{DS}/labelsTr'))
print('imagesTr:', len(imgs), '| labelsTr:', len(labs))
print('imagesTs:', len(os.listdir(f"{DS}/imagesTs")), '| labelsTs:', len(os.listdir(f"{DS}/labelsTs")))

ex = imgs[0].replace('_0000.nii.gz','')
a = nib.load(f'{DS}/imagesTr/{ex}_0000.nii.gz')
b = nib.load(f'{DS}/labelsTr/{ex}.nii.gz')
print('\nprimer:', ex)
print('  slika oblik :', a.shape)
print('  maska oblik :', b.shape, '| vrednosti u maski:', np.unique(np.asarray(b.dataobj)))
assert a.shape == b.shape, 'Oblik slike i maske se ne poklapa!'
print('\nOK — format izgleda ispravno.')


imagesTr: 178 | labelsTr: 178
imagesTs: 37 | labelsTs: 37

primer: mls_P10_T1
  slika oblik : (182, 218, 182)
  maska oblik : (182, 218, 182) | vrednosti u maski: [0. 1.]

OK — format izgleda ispravno.


## Ćelija 5 — nnU-Net: planiranje i pretprocesiranje
nnU-Net sam analizira podatke i pravi plan (rezolucija, veličina patcha, normalizacija).

In [ ]:
!nnUNetv2_plan_and_preprocess -d 1 --verify_dataset_integrity

Fingerprint extraction...
Dataset001_MsBrain
Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> reader/writer

####################
verify_dataset_integrity Done. 
If you didn't see any error messages then your dataset is most likely OK!
####################

Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> reader/writer
Extracting dataset fingerprint: 100% 178/178 [01:18<00:00,  2.27it/s]
Experiment planning...

############################
INFO: You are using the old nnU-Net default planner. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Dropping 3d_lowres config because the image size difference to 3d_fullres is too small. 3d_fullres: [148.5 187.5 150.5], 3d_lowres: [148, 188, 150]
2D U-Net configuration:
{'data_identifier': 'nnUNetPlans_2d', 'preprocessor_name': 'DefaultPreprocessor', 'batch_siz

## Ćelija 6 — TRENIRANJE


In [ ]:
import os

# putanja do checkpointa
rez = f"{os.environ['nnUNet_results']}/Dataset001_MsBrain/nnUNetTrainer_100epochs__nnUNetPlans__3d_fullres/fold_0"
nastavi = "--c" if os.path.exists(f"{rez}/checkpoint_latest.pth") else ""

print("Nastavljam od checkpointa." if nastavi else "Krećem trening od nule.")

!nnUNetv2_train 1 3d_fullres 0 -tr nnUNetTrainer_100epochs --npz {nastavi}

Nastavljam od checkpointa.

############################
INFO: You are using the old nnU-Net default plans. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Using device: cuda:0

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

2026-10-22 08:21:10.945042: Using torch.compile...
2026-10-22 08:21:21.355457: do_dummy_2d_data_aug: False
2026-10-22 08:21:21.360782: Creating new 5-fold cross-validation split...
2026-10-22 08:21:21.368949: Desired fold for training: 0
2026-10-22 08:21:21.371743: Thi